In [17]:
!pip install -q -U "transformers>=4.40.0" "torch>=2.2.0" "torchvision" "sentence-transformers" "accelerate" "bitsandbytes" "llama-cpp-python" "psutil"

import torch
if torch.cuda.is_available():
    print(f" GPU Ready: {torch.cuda.get_device_name(0)}")

 GPU Ready: Tesla T4


In [18]:
import time
import os
import csv
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
from llama_cpp import Llama
from sentence_transformers import SentenceTransformer, util

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
FINE_TUNED_PATH = "/content/drive/MyDrive/WEEK8_LLM/models/merged"
GGUF_PATH = "/content/drive/MyDrive/WEEK8_LLM/models/gguf/model.gguf"
MAX_NEW_TOKENS = 128

embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")

EVAL_PROMPTS = {
    "qa": "Write a Python function to check if a number is prime.",
    "reasoning": "Explain why a dictionary is faster than a list for lookups in Python.",
    "extraction": "Extract the function names from: def add(a,b): return a+b\ndef sub(a,b): return a-b"
}

GROUND_TRUTH = [
    "def is_prime(n): return n > 1 and all(n % i for i in range(2, int(n**0.5) + 1))",
    "Dictionaries use hash tables for O(1) complexity, while lists use linear search O(n).",
    "add, sub"
]

def semantic_accuracy(preds, refs):
    p_emb = embedder.encode(preds, convert_to_tensor=True)
    r_emb = embedder.encode(refs, convert_to_tensor=True)
    sims = util.cos_sim(p_emb, r_emb)
    return round(sims.diag().mean().item(), 3)

def get_vram_mb():
    return round(torch.cuda.max_memory_allocated() / 1024**2, 2) if torch.cuda.is_available() else 0

print(" Setup Complete. Ready for Day 4 Benchmarking.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Setup Complete. Ready for Day 4 Benchmarking.


In [5]:
!pip install -q sentencepiece

In [19]:
import gc
import time
import os
import torch
import csv
from transformers import AutoTokenizer, AutoModelForCausalLM
from llama_cpp import Llama

# 1. Configuration (Ensure paths match your Drive)
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
FINE_TUNED_PATH = "/content/quantized/model-fp16"
GGUF_PATH = "/content/quantized/model.gguf"

results = []

# 2. Benchmark Loop
configs = [
    {"name": "Base-Model", "path": BASE_MODEL, "engine": "transformers"},
    {"name": "Fine-Tuned", "path": FINE_TUNED_PATH, "engine": "transformers"},
    {"name": "GGUF-Q8", "path": GGUF_PATH, "engine": "llama.cpp"}
]

for cfg in configs:
    print(f"\n Benchmarking: {cfg['name']}")

    if cfg['engine'] == "transformers":
        # use_fast=False requires sentencepiece (Step 1)
        # padding_side='left' is required for proper batch generation in decoder-only models
        tokenizer = AutoTokenizer.from_pretrained(cfg['path'], use_fast=False)
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = 'left'

        model = AutoModelForCausalLM.from_pretrained(
            cfg['path'],
            torch_dtype=torch.float16,
            device_map="auto"
        )

        # Multi-prompt Batch Inference
        prompts = list(EVAL_PROMPTS.values())
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")

        torch.cuda.reset_peak_memory_stats()
        start = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                pad_token_id=tokenizer.eos_token_id
            )
        end = time.time()

        res_texts = [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]
        acc = semantic_accuracy(res_texts, GROUND_TRUTH)
        vram = get_vram_mb()

        # Cleanup to prevent Out-of-Memory (OOM)
        del model
        del tokenizer
        gc.collect()
        torch.cuda.empty_cache()

    else:
        # GGUF via llama.cpp (CPU)
        llm = Llama(model_path=cfg['path'], n_ctx=2048, verbose=False)
        start = time.time()
        res_texts = [llm(p, max_tokens=MAX_NEW_TOKENS)["choices"][0]["text"] for p in EVAL_PROMPTS.values()]
        end = time.time()
        acc = semantic_accuracy(res_texts, GROUND_TRUTH)
        vram = 0

    duration = end - start
    tokens = sum(len(t.split()) for t in res_texts)

    metrics = {
        "model": cfg['name'],
        "engine": cfg['engine'],
        "tokens_per_sec": round(tokens/duration, 2),
        "latency_sec": round(duration, 2),
        "vram_mb": vram,
        "accuracy": acc
    }
    results.append(metrics)
    print(metrics)

# Save deliverables to CSV
os.makedirs("/content/benchmarks", exist_ok=True)
with open("/content/benchmarks/results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print("\n Benchmark report saved: /content/benchmarks/results.csv")


 Benchmarking: Base-Model


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'model': 'Base-Model', 'engine': 'transformers', 'tokens_per_sec': 37.5, 'latency_sec': 4.21, 'vram_mb': 2543.13, 'accuracy': 0.719}

 Benchmarking: Fine-Tuned


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'model': 'Fine-Tuned', 'engine': 'transformers', 'tokens_per_sec': 63.08, 'latency_sec': 4.47, 'vram_mb': 2542.7, 'accuracy': 0.72}

 Benchmarking: GGUF-Q8
{'model': 'GGUF-Q8', 'engine': 'llama.cpp', 'tokens_per_sec': 3.35, 'latency_sec': 59.13, 'vram_mb': 0, 'accuracy': 0.731}

 Benchmark report saved: /content/benchmarks/results.csv


In [20]:
from transformers import TextStreamer

tokenizer = AutoTokenizer.from_pretrained(FINE_TUNED_PATH, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(
    FINE_TUNED_PATH,
    torch_dtype=torch.float16,
    device_map="auto"
)
streamer = TextStreamer(tokenizer, skip_prompt=True)

prompt = "Write a Python function to find the maximum number in a list."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

print(f"Prompt: {prompt}\nResponse: ")
_ = model.generate(**inputs, streamer=streamer, max_new_tokens=128)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Prompt: Write a Python function to find the maximum number in a list.
Response: 


# Function to find the maximum number in a list
def find_max(lst):
   max_num = lst[0]
   for num in lst:
       if num > max_num:
           max_num = num
   return max_num

# Call the function
max_num = find_max([1, 2, 3, 4, 5])
print(max_num) # Output: 5

# Output: 5
max_num = find_max([1, 2, 3, 4, 
